In [1]:
import subprocess, sys

BITSANDBYTES_PIN = "0.49.2"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)
print("profiling pins installed (no vLLM today)")


installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
profiling pins installed (no vLLM today)


In [2]:
import gc, torch
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def reserved_mb():
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 2)

def free_vram():
    gc.collect()
    torch.cuda.empty_cache()

samples = []
for i in range(5):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.float16, device_map="cuda"
    )
    after_load = reserved_mb()

    del model
    free_vram()

    after_unload = reserved_mb()
    samples.append({
        "cycle": i,
        "after_load_mb": round(after_load, 1),
        "after_unload_mb": round(after_unload, 1)
    })
    print(samples[-1])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'cycle': 0, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 1, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 2, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 3, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}
{'cycle': 4, 'after_load_mb': 3134.0, 'after_unload_mb': 0.0}


In [3]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")
tok_ids = torch.randint(0, 1000, (1, 64)).to("cuda")

leaked_outputs = []   # this is the leak
leak_samples = []
for i in range(20):
    # NOTE: no torch.no_grad() here, on purpose
    out = model(tok_ids)
    leaked_outputs.append(out.logits)   # holds the autograd graph alive too
    leak_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})
    if i % 5 == 0:
        print(leak_samples[-1])

{'iter': 0, 'reserved_mb': 3254.0}
{'iter': 5, 'reserved_mb': 4294.0}
{'iter': 10, 'reserved_mb': 5354.0}
{'iter': 15, 'reserved_mb': 6414.0}


In [4]:
import numpy as np

def detect_leak(samples_mb, slope_threshold_mb_per_iter=1.0):
    """samples_mb: list of reserved-memory readings, one per iteration."""
    x = np.arange(len(samples_mb))
    y = np.array(samples_mb)
    slope, intercept = np.polyfit(x, y, 1)
    leaking = slope > slope_threshold_mb_per_iter
    return {
        "slope_mb_per_iter": round(float(slope), 3),
        "threshold_mb_per_iter": slope_threshold_mb_per_iter,
        "leaking": bool(leaking),
        "n_samples": len(samples_mb),
    }

leak_result = detect_leak([s["reserved_mb"] for s in leak_samples])
print(leak_result)
assert leak_result["leaking"], "expected the Step 2 loop to be flagged as leaking"

{'slope_mb_per_iter': 211.248, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


In [5]:
del leaked_outputs
del model
free_vram()

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda"
)

fixed_samples = []
with torch.no_grad():
    for i in range(20):
        out = model(tok_ids)
        _ = out.logits.sum().item()
        fixed_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})

fixed_result = detect_leak([s["reserved_mb"] for s in fixed_samples])
print(fixed_result)
assert not fixed_result["leaking"], "still leaking after the fix -- check both causes were removed"

del model
free_vram()

{'slope_mb_per_iter': -0.0, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}


In [6]:
import json

report = {
    "reload_loop_baseline": samples,
    "leaky_run": leak_result,
    "fixed_run": fixed_result,
    "leaky_samples": [s["reserved_mb"] for s in leak_samples],
    "fixed_samples": [s["reserved_mb"] for s in fixed_samples],
}

with open("leak_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "reload_loop_baseline": [
    {
      "cycle": 0,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 1,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 2,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 3,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    },
    {
      "cycle": 4,
      "after_load_mb": 3134.0,
      "after_unload_mb": 0.0
    }
  ],
  "leaky_run": {
    "slope_mb_per_iter": 211.248,
    "threshold_mb_per_iter": 1.0,
    "leaking": true,
    "n_samples": 20
  },
  "fixed_run": {
    "slope_mb_per_iter": -0.0,
    "threshold_mb_per_iter": 1.0,
    "leaking": false,
    "n_samples": 20
  },
  "leaky_samples": [
    3254.0,
    3462.0,
    3670.0,
    3878.0,
    4086.0,
    4294.0,
    4522.0,
    4730.0,
    4938.0,
    5146.0,
    5354.0,
    5562.0,
    5790.0,
    5998.0,
    6206.0,
    6414.0,
    6622.0,
    6830.0,
    7058.0,
  

In [7]:
import json, os
from typing import NoReturn

LEAK_FLOOR_MB_PER_ITER = 5.0    # the deliberate leak retains logits + graph; a
                                # real reproduction climbs far faster than this
FIXED_CEIL_MB_PER_ITER = 1.0    # allocator drift, not a leak
MIN_SAMPLES = 15
DRIFT_CEIL_MB_PER_CYCLE = 100.0  # reload-loop baseline may drift, not climb


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def _fail(reason) -> NoReturn:
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()


def slope(ys):
    """Least-squares slope of ys against 0..n-1, no numpy."""
    n = len(ys)
    xs = range(n)
    mx = (n - 1) / 2.0
    my = sum(ys) / n
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    den = sum((x - mx) ** 2 for x in xs)
    return num / den if den else 0.0


def main():
    if not os.path.isfile("leak_report.json"):
        _fail("leak_report.json not found; run Step 5 first")
    try:
        with open("leak_report.json") as f:
            r = json.load(f)
    except json.JSONDecodeError as e:
        _fail("leak_report.json is not valid JSON: %s" % e)

    for key in ("reload_loop_baseline", "leaky_run", "fixed_run",
                "leaky_samples", "fixed_samples"):
        if key not in r:
            _fail("missing key '%s' (Step 5 in the current lab writes raw "
                  "samples too; re-run it)" % key)

    base = r["reload_loop_baseline"]
    if not isinstance(base, list) or len(base) < 5:
        _fail("reload_loop_baseline needs the 5 control cycles")
    unloads = []
    for row in base:
        try:
            if row["after_unload_mb"] >= row["after_load_mb"]:
                _fail("cycle %s: after_unload >= after_load; unload freed "
                      "nothing" % row.get("cycle"))
            unloads.append(float(row["after_unload_mb"]))
        except (KeyError, TypeError):
            _fail("baseline rows need after_load_mb and after_unload_mb")
    base_slope = slope(unloads)
    if base_slope > DRIFT_CEIL_MB_PER_CYCLE:
        _fail("control loop itself climbs %.1f MB/cycle; the notebook leaked "
              "before Step 2 (restart the runtime and rerun)" % base_slope)

    for name, samples_key, run_key in (("leaky", "leaky_samples", "leaky_run"),
                                       ("fixed", "fixed_samples", "fixed_run")):
        ys = r[samples_key]
        if not isinstance(ys, list) or len(ys) < MIN_SAMPLES:
            _fail("%s run has %s samples, need >= %d"
                  % (name, len(ys) if isinstance(ys, list) else "no", MIN_SAMPLES))
        if not all(isinstance(y, (int, float)) for y in ys):
            _fail("%s_samples must be numbers (MB readings)" % name)
        my_slope = slope(ys)
        claimed = r[run_key]
        their_slope = claimed.get("slope_mb_per_iter")
        if not isinstance(their_slope, (int, float)) or abs(their_slope - my_slope) > max(0.5, abs(my_slope) * 0.15):
            _fail("%s run: detector says %.3f MB/iter, an independent refit of "
                  "the same samples says %.3f; the detector is not reading its "
                  "own data" % (name, their_slope or float("nan"), my_slope))
        if name == "leaky":
            if my_slope < LEAK_FLOOR_MB_PER_ITER:
                _fail("leaky run climbs only %.2f MB/iter; the deliberate leak "
                      "was not actually reproduced (both causes in Step 2?)" % my_slope)
            if not claimed.get("leaking"):
                _fail("leaky run climbs %.1f MB/iter but the detector says "
                      "leaking=false" % my_slope)
        else:
            if my_slope > FIXED_CEIL_MB_PER_ITER:
                _fail("fixed run still climbs %.2f MB/iter; one of the two "
                      "causes survived the fix" % my_slope)
            if claimed.get("leaking"):
                _fail("fixed run is flat (%.2f MB/iter) but the detector says "
                      "leaking=true; its threshold is misread" % my_slope)

    print("refit slopes agree with the detector; leak reproduced then removed")
    print("GREEN CHECK: PASS")


if __name__ == "__main__":
    try:
        main()
    except _Stop:
        raise SystemExit(1)

refit slopes agree with the detector; leak reproduced then removed
GREEN CHECK: PASS
